# How to use ri-layercake

ri-layercake turns reinsurance programmes into presentation-ready towers and native, editable PowerPoint slides.
Every cell on this page is live: edit it and press **Run**. The code runs in your browser, on the same engine as the
Playground.

## 1. Install

```
pip install ri-layercake
```

One install gives you everything: PowerPoint, Excel and CSV input, DataFrames and matplotlib plots.

## 2. Your first tower

Shorthand is the quickest way in: `10 ret` is a retention, `40 xs 10` is a £40m layer excess of £10m, and
`@ 75%` means 75% placed. A bare tower in a cell draws itself with its table.

In [ ]:
import ri_layercake as lc

tower = lc.Tower.from_text(
    "10 ret, 40 xs 10, 50 xs 50 @ 75%, 100 xs 100",
    currency="GBP", units="m", title="Property cat 2027",
)
tower

## 3. The table and totals

`tower.table()` returns the same rows the slide shows, top of the tower first, as a DataFrame. Price a layer by
premium, rate on line (`rol`) or adjusted rate on EPI (`rate`); the other two are derived.

In [ ]:
tower = lc.Tower.from_text("epi 120, 10 ret, 40 xs 10 prem 4, 50 xs 50 @ 75% rate 2.5%, 100 xs 100 rol 2.5%")
tower.table()

## 4. Validation warnings and fixes

Gaps, overlaps and unplaced shares are often legitimate, so they are warnings, never blockers. Each warning carries
suggested fixes you can apply.

In [ ]:
gappy = lc.Tower.from_text("10 ret, 40 xs 10, 50 xs 60")
for d in gappy.validate():
    print(d.severity, d.message)
    print("  fixes:", [f.label for f in d.fixes])

In [ ]:
fixed = gappy.apply(gappy.validate()[0].fixes[0])   # Lower next attachment
fixed.validate(), fixed.to_text()

## 5. Building from Excel and DataFrames

`Tower.from_excel` reads the first sheet (or `sheet=`), finds the header row, and maps columns by name: limit,
excess, deductible, share, premium, rate on line and their usual synonyms. This cell writes a small workbook first so
it runs anywhere.

In [ ]:
from openpyxl import Workbook

wb = Workbook()
ws = wb.active
ws.append(["Property cat programme 2027"])
ws.append([])
ws.append(["Layer", "Limit (£m)", "Excess (£m)", "% Placed", "Premium (£m)"])
ws.append(["Layer 1", 40, 10, "100%", 4.0])
ws.append(["Layer 2", 50, 50, "75%", 3.0])
ws.append(["Total", 90, None, None, 7.0])
wb.save("programme.xlsx")

from_excel = lc.Tower.from_excel("programme.xlsx")
from_excel.to_text()

In [ ]:
import pandas as pd

df = pd.DataFrame({"limit": [40, 50], "attachment": [10, 50], "placed": [1, 0.8]})
lc.Tower.from_dataframe(df).to_text()

## 6. Editing: add, split, merge

Towers are immutable: every edit returns a new tower, so you can keep the old one. Amounts are in the tower's units.

In [ ]:
t = lc.Tower.from_text("10 ret, 40 xs 10, 100 xs 50")
t = t.add_xl(150, 150, placed=0.6, name="Top")
t = t.split("L2", at=100)         # 100 xs 50 becomes 50 xs 50 and 50 xs 100
print(t.to_text())
t = t.merge("L2", "L3")           # and back again
print(t.to_text())

## 7. Views: schematic, linear and log

Schematic gives every layer the same height, like most broker slides. Linear makes heights proportional to the
limit. Log keeps a thin retention visible under a tall tower.

In [ ]:
from ri_layercake import gallery

tall = gallery.load("thin_retention")
tall.show(scale="log")

In [ ]:
tall.show(scale="linear", palette="slate", table="right")

## 8. Drop a loss

`drop_loss` sends a single loss up the tower: recovery per layer on the 100% and placed bases, whether the layer is
exhausted, and reinstatement premium pro rata to amount.

In [ ]:
cat = gallery.load("property_cat")
result = cat.drop_loss(120)
result

In [ ]:
totals = result.to_dict()["totals"]
print("recovered", totals["recovered"], "retained", totals["retained"])
cat.show(scale="linear", loss=120)

## 9. Quota share: stacked or side by side

Add a QS with `qs 20% limit 50` (the limit is usually the CRM team's limit for your largest counterparty).
**Stacked** (the default): the QS cedes first and the net passes through every XL layer. For a 50m exposure with a
20% QS, the QS takes 10m and the remaining 40m runs through the XL layers.

In [ ]:
qs_tower = lc.Tower.from_text("epi 120, qs 20% limit 50 agg 300 comm 30%, 5 xs 5, 10 xs 10, 30 xs 20")
r = qs_tower.exposure(50)
totals = r.to_dict()["totals"]
print("QS ceded", totals["qs_ceded"], "· net into the XL", totals["net_of_qs"])
r

**Side by side**: each XL attachment and limit is lowered by the QS cession and applied to the net retained, so a
40 xs 10 behind a 20% QS works as 32 xs 8.

In [ ]:
side = qs_tower.programme(arrangement="side_by_side")
side.exposure(50)

In [ ]:
side.show(scale="linear")

## 10. Stop loss, QS terms and no claims bonus

A stop loss is set as % of EPI (`sl 20% xs 80%`) and works on the year's aggregate loss after the QS. QS terms
cover ceding commission, a sliding scale, reinsurer management expenses and profit commission; XL layers can carry
a no claims bonus.

In [ ]:
prog = (
    lc.Tower.from_text("epi 100, qs 20% limit 50 agg 300, 5 xs 5 rate 2% ncb 10%, sl 20% xs 80% rate 1.5%")
    .update("Q1", terms=lc.Terms(
        sliding_scale=lc.SlidingScale(min=0.25, max=0.35, lr_low=0.5, lr_high=0.7),
        management_expenses=0.05, profit_commission=0.2,
    ))
)
year = prog.aggregate(110)          # a 110m aggregate loss year
year.to_dict()["qs_terms"], year.to_dict()["rows"]

## 11. Portfolio trapped exposure

Pass a list of risks, a pasted table or a CSV/Excel path with an exposure (sum insured) column. Each risk's trapped
exposure in a layer is `median(0, exposure − attachment, limit)`.

In [ ]:
book = [("Warehouse", 1.5), ("Hotel", 12), ("Factory", 18.5), ("Data centre", 48), ("Airport", 85)]
port = gallery.load("per_risk_xol").portfolio(book)
import pandas as pd
pd.DataFrame(port.summary())

## 12. Premium walk and plots

The premium walk goes from gross EPI to the result and margin after reinsurance. Steps marked * are approximate;
every step carries its formula. `tower.plot()` and `walk.plot()` draw with matplotlib.

In [ ]:
full = gallery.load("full_programme")
walk = full.premium_walk()
walk

In [ ]:
walk.plot();

In [ ]:
full.plot();

## 13. Export to PowerPoint

`to_pptx` writes one slide with native shapes and a native table in theme colours, so pasting it into a client deck
recolours it to that deck's theme. Pass `template=` with your own .pptx or .potx to use its theme, fonts and title
layout. In the browser, each new file gets a download chip.

In [ ]:
cat.to_pptx("renewal_2027.pptx", view=lc.View(scale="schematic", palette="ocean"))

In [ ]:
from pptx import Presentation

house = Presentation()                       # stand-in for your house template
house.slide_width, house.slide_height = 12_192_000, 6_858_000
house.save("house_style.pptx")

cat.to_pptx("renewal_house_style.pptx", template="house_style.pptx", table_slide=True)

## 14. Batch: one slide per programme

A short loop turns a workbook of programmes into a folder of slides. This cell builds a three-sheet workbook, then
exports one PowerPoint per sheet.

In [ ]:
programmes = {
    "Marine": [["Limit", "Excess"], [20, 5], [25, 25], [50, 50]],
    "Casualty": [["Limit", "Excess", "Share"], [5, 5, "100%"], [10, 10, "100%"], [20, 20, "80%"]],
    "Property": [["Layer", "Share"], ["40 xs 10", "100%"], ["50 xs 50", "90%"], ["100 xs 100", "70%"]],
}
wb = Workbook()
wb.remove(wb.active)
for name, rows in programmes.items():
    ws = wb.create_sheet(name)
    for r in rows:
        ws.append(r)
wb.save("programmes.xlsx")

for name in programmes:
    t = lc.Tower.from_excel("programmes.xlsx", sheet=name, title=f"{name} XL 2027")
    t.to_pptx(f"{name.lower()}.pptx")
    print(f"{name:9} {t.to_text()}")

## 15. The command line

The package installs an `ri-layercake` command:

```
ri-layercake tower programme.xlsx --out renewal.pptx
ri-layercake tower "10 ret, 40 xs 10, 50 xs 50" --scale linear --svg tower.svg
ri-layercake check programme.xlsx          # prints diagnostics, exit code 1 on errors
ri-layercake loss "40 xs 10, 50 xs 50" 75  # drop-a-loss table
```

The same commands are available from Python:

In [ ]:
from ri_layercake.cli import main

main(["loss", "10 ret, 40 xs 10, 50 xs 50 @ 75%", "75"])

Outputs are illustrative, based on the inputs provided, and not pricing or actuarial advice.